# **League of Legends. Этап 2 - Transform + Этап 3 - Load (загрузка в хранилище)** 

23.06.2026

**Задача:** 
1. Анализ и преобразование полученных по API на предыдущем этапе данных
2. Последующая загрузка данных в БД **Supabase.**
3. Создание витрин данных для дашборда в БД Supabase.

## Импорт библиотек и предварительные настройки

In [1]:
import pandas as pd  # для работы с таблицами
import re # для работы с регулярными выражениями

In [2]:
# Для работы с БД
from sqlalchemy import create_engine, text 
from dotenv import load_dotenv
from pathlib import Path
import os

In [3]:
# Настройки отображения DataFrame
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)

In [4]:
# базовый путь
base_dir = Path(os.getcwd()) 

In [5]:
def load_data_to_df(file_name, folder):
    """
    Функция загружает файл file_name из папки folder в датафрейм df
    """
    full_path = folder / file_name
    try:
        df = pd.read_csv(full_path, encoding='utf-8', encoding_errors='ignore')
        display(f"Файл загружен успешно: {file_name} | Строк: {df.shape[0]}")
        return df
    except Exception as e:
        print(f"Ошибка при загрузке: {file_name}: {e}")
        return None

## Первичный анализ и предобработка извлеченых данных

### Все игроки лиг

In [109]:
# Загружаем 
df_players = load_data_to_df('all_players_data.csv', base_dir)

'Файл загружен успешно: all_players_data.csv | Строк: 21631'

In [110]:
df_players.head()  

,puuid,leaguePoints,rank,wins,losses,veteran,inactive,freshBlood,hotStreak,league_type,region
0,7qDgswqpOmHpVyAQO-085TwvVSp5kuojnI7ehOzZiBQX4J...,3814,I,787,643,True,False,False,False,challenger,euw1
1,D8z6JW2Ea_FfL6zRR98wcFmaMpK1KZRbYBGS3pIySYqNC0...,3722,I,263,201,True,False,False,False,challenger,euw1
2,Sb7usV3fHduwHxZoXjwHdaVxF1T2LhaBHejvJdqOvRhZv_...,3598,I,749,628,True,False,False,False,challenger,euw1
3,ZIemTzkIPML6oScobsLKDZZCmGXzGA9yHbJAymoaqRNFop...,3594,I,386,314,True,False,False,False,challenger,euw1
4,tlYvTzzoBLVbP6ZISTMT9V8mfibz4nNJVHkWbMSK5xlzF6...,3492,I,407,271,True,False,False,False,challenger,euw1


In [111]:
# анализируем структуру датафрейма
df_players.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21631 entries, 0 to 21630
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   puuid         21631 non-null  object
 1   leaguePoints  21631 non-null  int64 
 2   rank          21631 non-null  object
 3   wins          21631 non-null  int64 
 4   losses        21631 non-null  int64 
 5   veteran       21631 non-null  bool  
 6   inactive      21631 non-null  bool  
 7   freshBlood    21631 non-null  bool  
 8   hotStreak     21631 non-null  bool  
 9   league_type   21631 non-null  object
 10  region        21631 non-null  object
dtypes: bool(4), int64(3), object(4)
memory usage: 1.2+ MB


In [112]:
#Приводим все столбцы к стилю snake case. 
# re.sub(pattern, replacement, string) — замена текста
df_players = df_players .rename(columns=lambda x: re.sub(r'(?<!^)(?=[A-Z])', '_', x).lower())
#проверяем корректность переименования
df_players.columns

Index(['puuid', 'league_points', 'rank', 'wins', 'losses', 'veteran',
       'inactive', 'fresh_blood', 'hot_streak', 'league_type', 'region'],
      dtype='object')

In [113]:
# Дубликаты
#проверим на полные дубликаты 
display(f'Полных дубликатов {df_players.duplicated().sum()} строк')
#проверим на дубликаты по puuid
display(f'Дубликатов по puuid {df_players.duplicated(subset = ["puuid"]).sum()} строк')

'Полных дубликатов 0 строк'

'Дубликатов по puuid 1 строк'

In [114]:
# Смотрим строки с дубликатами
duplicates = df_players.loc[df_players.duplicated(subset=["puuid"], keep=False)]
display(duplicates)

,puuid,league_points,rank,wins,losses,veteran,inactive,fresh_blood,hot_streak,league_type,region
7066,3CpYfP2MSwmsgofBURsEiqqyNZHJqHBmkQ5bQ-Bq92aWzC...,469,I,88,84,True,False,False,False,master,euw1
16364,3CpYfP2MSwmsgofBURsEiqqyNZHJqHBmkQ5bQ-Bq92aWzC...,157,I,322,307,False,False,False,False,master,na1


Видим, что один игрок поменял регион, это допускается. Значит ключом к игроку является связка (puuid + region)

In [115]:
# Проверим на пустые строки
df_players.isna().sum()

puuid            0
league_points    0
rank             0
wins             0
losses           0
veteran          0
inactive         0
fresh_blood      0
hot_streak       0
league_type      0
region           0
dtype: int64

In [117]:
# Преобразуем для однообразия имя сервера и название лиги в верхний регистр
df_players['league_type'] = df_players['league_type'].str.upper()
df_players['region'] = df_players['region'].str.upper()

### ТОП-игроки лиг

In [119]:
# Загружаем 
df_top_players = load_data_to_df('top_players.csv', base_dir)

'Файл загружен успешно: top_players.csv | Строк: 300'

In [120]:
df_top_players.head()   

,puuid,leaguePoints,rank,wins,losses,veteran,inactive,freshBlood,hotStreak,league_type,region,total_games
0,7qDgswqpOmHpVyAQO-085TwvVSp5kuojnI7ehOzZiBQX4J...,3814,I,787,643,True,False,False,False,challenger,euw1,1430
1,D8z6JW2Ea_FfL6zRR98wcFmaMpK1KZRbYBGS3pIySYqNC0...,3722,I,263,201,True,False,False,False,challenger,euw1,464
2,Sb7usV3fHduwHxZoXjwHdaVxF1T2LhaBHejvJdqOvRhZv_...,3598,I,749,628,True,False,False,False,challenger,euw1,1377
3,ZIemTzkIPML6oScobsLKDZZCmGXzGA9yHbJAymoaqRNFop...,3594,I,386,314,True,False,False,False,challenger,euw1,700
4,tlYvTzzoBLVbP6ZISTMT9V8mfibz4nNJVHkWbMSK5xlzF6...,3492,I,407,271,True,False,False,False,challenger,euw1,678


In [121]:
# анализируем структуру датафрейма
df_top_players.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   puuid         300 non-null    object
 1   leaguePoints  300 non-null    int64 
 2   rank          300 non-null    object
 3   wins          300 non-null    int64 
 4   losses        300 non-null    int64 
 5   veteran       300 non-null    bool  
 6   inactive      300 non-null    bool  
 7   freshBlood    300 non-null    bool  
 8   hotStreak     300 non-null    bool  
 9   league_type   300 non-null    object
 10  region        300 non-null    object
 11  total_games   300 non-null    int64 
dtypes: bool(4), int64(4), object(4)
memory usage: 20.1+ KB


In [122]:
#Приводим все столбцы к стилю snake case. 
# re.sub(pattern, replacement, string) — замена текста
df_top_players = df_top_players .rename(columns=lambda x: re.sub(r'(?<!^)(?=[A-Z])', '_', x).lower())
#проверяем корректность переименования
df_top_players.columns

Index(['puuid', 'league_points', 'rank', 'wins', 'losses', 'veteran',
       'inactive', 'fresh_blood', 'hot_streak', 'league_type', 'region',
       'total_games'],
      dtype='object')

In [123]:
# Дубликаты
#проверим на полные дубликаты 
display(f'Полных дубликатов {df_top_players.duplicated().sum()} строк')
#проверим на дубликаты по puuid
display(f'Дубликатов по puuid {df_top_players.duplicated(subset = ["puuid"]).sum()} строк')

'Полных дубликатов 0 строк'

'Дубликатов по puuid 0 строк'

In [124]:
# Проверим на пустые строки
df_top_players.isna().sum()

puuid            0
league_points    0
rank             0
wins             0
losses           0
veteran          0
inactive         0
fresh_blood      0
hot_streak       0
league_type      0
region           0
total_games      0
dtype: int64

### Матчи

In [125]:
# Загружаем 
df_matches = load_data_to_df('matches_data.csv', base_dir)

'Файл загружен успешно: matches_data.csv | Строк: 31965'

In [126]:
df_matches.head()   

,match_id,game_name,game_creation,game_duration,game_version,game_mode,endOfGameResult,gameStartTimestamp,gameEndTimestamp,gameType,platform_id,queue_id
0,NA1_5569640145,teambuilder-match-5569640145,1779928905600,1641,16.10.776.5552,CLASSIC,GameComplete,1779928919031,1779930560112,MATCHED_GAME,NA1,420
1,EUW1_7858854154,teambuilder-match-7858854154,1779186460514,1236,16.10.776.5552,CLASSIC,GameComplete,1779186473836,1779187710175,MATCHED_GAME,EUW1,420
2,EUW1_7866192219,teambuilder-match-7866192219,1779755407447,1506,16.10.776.5552,CLASSIC,GameComplete,1779755423023,1779756929196,MATCHED_GAME,EUW1,420
3,EUW1_7863978356,teambuilder-match-7863978356,1779607251853,1486,16.10.776.5552,CLASSIC,GameComplete,1779607265724,1779608751707,MATCHED_GAME,EUW1,420
4,NA1_5570117033,teambuilder-match-5570117033,1779996545386,1464,16.11.781.1789,CHERRY,GameComplete,1779996560727,1779998024777,MATCHED_GAME,NA1,1750


In [127]:
# анализируем структуру датафрейма
df_matches.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31965 entries, 0 to 31964
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   match_id            31965 non-null  object
 1   game_name           31959 non-null  object
 2   game_creation       31965 non-null  int64 
 3   game_duration       31965 non-null  int64 
 4   game_version        31959 non-null  object
 5   game_mode           31959 non-null  object
 6   endOfGameResult     31965 non-null  object
 7   gameStartTimestamp  31965 non-null  int64 
 8   gameEndTimestamp    31965 non-null  int64 
 9   gameType            31959 non-null  object
 10  platform_id         31959 non-null  object
 11  queue_id            31965 non-null  int64 
dtypes: int64(5), object(7)
memory usage: 2.9+ MB


In [128]:
#Приводим все столбцы к стилю snake case. 
# re.sub(pattern, replacement, string) — замена текста
df_matches = df_matches.rename(columns=lambda x: re.sub(r'(?<!^)(?=[A-Z])', '_', x).lower())
#проверяем корректность переименования
df_matches.columns

Index(['match_id', 'game_name', 'game_creation', 'game_duration',
       'game_version', 'game_mode', 'end_of_game_result',
       'game_start_timestamp', 'game_end_timestamp', 'game_type',
       'platform_id', 'queue_id'],
      dtype='object')

In [129]:
# Дубликаты
#проверим на полные дубликаты 
display(f'Полных дубликатов {df_matches.duplicated().sum()} строк')
#проверим на дубликаты по ID матча
display(f'Дубликатов по ID матча {df_matches.duplicated(subset = ["match_id"]).sum()} строк')

'Полных дубликатов 0 строк'

'Дубликатов по ID матча 0 строк'

In [130]:
# Проверим на пустые строки
df_matches.isna().sum()

match_id                0
game_name               6
game_creation           0
game_duration           0
game_version            6
game_mode               6
end_of_game_result      0
game_start_timestamp    0
game_end_timestamp      0
game_type               6
platform_id             6
queue_id                0
dtype: int64

In [131]:
# Посмотрим на строки с пустыми колонками
df_matches.loc[df_matches.isna().any(axis=1)]

,match_id,game_name,game_creation,game_duration,game_version,game_mode,end_of_game_result,game_start_timestamp,game_end_timestamp,game_type,platform_id,queue_id
6853,EUW1_7858197708,NaN,0,0,NaN,NaN,Abort_Unexpected,0,1779126018864,NaN,NaN,0
14264,EUW1_7867534257,NaN,0,0,NaN,NaN,Abort_Unexpected,0,1779891511407,NaN,NaN,0
16679,EUW1_7864386655,NaN,0,0,NaN,NaN,Abort_Unexpected,0,1779636219449,NaN,NaN,0
20584,EUW1_7864708855,NaN,0,0,NaN,NaN,Abort_Unexpected,0,1779651658429,NaN,NaN,0
23096,EUW1_7864839258,NaN,0,0,NaN,NaN,Abort_Unexpected,0,1779656977997,NaN,NaN,0
30289,EUW1_7853435748,NaN,0,0,NaN,NaN,Abort_Unexpected,0,1778773842596,NaN,NaN,0


Видим, что длительность матча =0, это какой-то сбой. Удалим эти строки.

In [132]:
# Удалим пустые cтроки
df_matches = df_matches.dropna(subset=['platform_id'])
# Проверим 
df_matches.isna().sum()

match_id                0
game_name               0
game_creation           0
game_duration           0
game_version            0
game_mode               0
end_of_game_result      0
game_start_timestamp    0
game_end_timestamp      0
game_type               0
platform_id             0
queue_id                0
dtype: int64

In [133]:
# Преобразуем даты создания/начала/окончания матча в милисекундах в дату и запишем в новые столбцы
df_matches['game_creation_dt'] = pd.to_datetime(df_matches['game_creation'], unit='ms')
df_matches['game_start_dt'] = pd.to_datetime(df_matches['game_start_timestamp'], unit='ms')
df_matches['game_end_dt'] = pd.to_datetime(df_matches['game_end_timestamp'], unit='ms')

In [134]:
# Смотрим уникальные значения в столбцах
display(f"Платформы: {df_matches['platform_id'].unique()}")
display(f"Версия: {df_matches['game_version'].unique()}")
display(f"Режим: {df_matches['game_mode'].unique()}")
display(f"Результат: {df_matches['end_of_game_result'].unique()}")
display(f"Тип игры: {df_matches['game_type'].unique()}")
display(f"Очередь: {df_matches['queue_id'].unique()}")

"Платформы: ['NA1' 'EUW1']"

"Версия: ['16.10.776.5552' '16.11.781.1789' '16.10.775.1511' '16.9.772.8292'\n '16.9.772.1032' '16.11.779.4239' '16.9.771.8383']"

"Режим: ['CLASSIC' 'CHERRY' 'ARAM' 'SWIFTPLAY']"

"Результат: ['GameComplete']"

"Тип игры: ['MATCHED_GAME']"

'Очередь: [ 420 1750 1700  440  450  400  480]'

In [135]:
# Длины текстовыех полей - чтобы в дальнейшем настроить размер VARCHAR в Supabase
display(f"Максимальная длина в колонке 'match_id': {df_matches['match_id'].str.len().max()}")
display(f"Максимальная длина в колонке 'game_version': {df_matches['game_version'].str.len().max()}")
display(f"Максимальная длина в колонке 'game_mode': {df_matches['game_mode'].str.len().max()}")
display(f"Максимальная длина в колонке 'platform_id': {df_matches['platform_id'].str.len().max()}")

"Максимальная длина в колонке 'match_id': 15"

"Максимальная длина в колонке 'game_version': 14"

"Максимальная длина в колонке 'game_mode': 9"

"Максимальная длина в колонке 'platform_id': 4"

In [136]:
# длительность матча
df_matches['game_duration'].describe()

count    31959.000000
mean      1598.050815
std        402.835759
min         65.000000
25%       1394.000000
50%       1603.000000
75%       1844.000000
max       3345.000000
Name: game_duration, dtype: float64

### Данные по игрокам в матчах 

In [137]:
# Загружаем 
df_players_matches = load_data_to_df('players_data.csv', base_dir)

'Файл загружен успешно: players_data.csv | Строк: 343524'

In [138]:
df_players_matches.head() 

,match_id,puuid,riotIdGameName,riotIdTagline,kills,deaths,assists,win,teamPosition,goldEarned,goldSpent,team_id,champion_id,championName,timePlayed
0,NA1_5569640145,7DSjgcHXZIecO8UR-GZrheKhQ8BV6oL4evWTcnPHulcJb0...,Chlodovech,MKY,4,3,8,True,TOP,12422,9733,100,114,Fiora,1641
1,NA1_5569640145,202lKXdX28Ws9YXws0NDQvDMS2gxEeN9c7R1RhKijDpYIu...,ken,IPT,12,6,10,True,JUNGLE,15706,14100,100,56,Nocturne,1641
2,NA1_5569640145,MGS0DwykFkw-6rBjPIhecgXmKkZw_T79Lah7rg3NaCDFZK...,Blue,0089,2,4,10,True,MIDDLE,10775,9325,100,69,Cassiopeia,1641
3,NA1_5569640145,tbpCvGCRlk0RqAELk6U1w_UqTDn2PoXs4nvhy-YvRQB_wI...,GREYH0UND,NA1,8,8,3,True,BOTTOM,13407,12900,100,145,Kaisa,1641
4,NA1_5569640145,gZ45WAgc-PyTPbQVT4OUzg-r8GV2EwKpt2yAxoFjbKDdTr...,3uphoria,0622,3,3,13,True,UTILITY,9319,7300,100,111,Nautilus,1641


In [139]:
# анализируем структуру датафрейма
df_players_matches.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 343524 entries, 0 to 343523
Data columns (total 15 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   match_id        343524 non-null  object
 1   puuid           343524 non-null  object
 2   riotIdGameName  343522 non-null  object
 3   riotIdTagline   343507 non-null  object
 4   kills           343524 non-null  int64 
 5   deaths          343524 non-null  int64 
 6   assists         343524 non-null  int64 
 7   win             343524 non-null  bool  
 8   teamPosition    288259 non-null  object
 9   goldEarned      343524 non-null  int64 
 10  goldSpent       343524 non-null  int64 
 11  team_id         343524 non-null  int64 
 12  champion_id     343524 non-null  int64 
 13  championName    343524 non-null  object
 14  timePlayed      343524 non-null  int64 
dtypes: bool(1), int64(8), object(6)
memory usage: 37.0+ MB


In [140]:
#Приводим все столбцы к стилю snake case. 
# re.sub(pattern, replacement, string) — замена текста
df_players_matches = df_players_matches.rename(columns=lambda x: re.sub(r'(?<!^)(?=[A-Z])', '_', x).lower())
#проверяем корректность переименования
df_players_matches.columns

Index(['match_id', 'puuid', 'riot_id_game_name', 'riot_id_tagline', 'kills',
       'deaths', 'assists', 'win', 'team_position', 'gold_earned',
       'gold_spent', 'team_id', 'champion_id', 'champion_name', 'time_played'],
      dtype='object')

In [141]:
# Дубликаты
#проверим на полные дубликаты 
display(f'Полных дубликатов {df_players_matches.duplicated().sum()} строк')
#проверим на дубликаты по связке ID матча + puuid игрока
display(f'Дубликатов по связке ID матча + puuid игрока: {df_players_matches.duplicated(subset = ["match_id","puuid"]).sum()} строк')

'Полных дубликатов 0 строк'

'Дубликатов по связке ID матча + puuid игрока: 0 строк'

In [142]:
# Проверим на пустые строки
df_players_matches.isna().sum()

match_id                 0
puuid                    0
riot_id_game_name        2
riot_id_tagline         17
kills                    0
deaths                   0
assists                  0
win                      0
team_position        55265
gold_earned              0
gold_spent               0
team_id                  0
champion_id              0
champion_name            0
time_played              0
dtype: int64

In [143]:
# Посмотрим на строку с пустым именем игрока
df_players_matches.loc[df_players_matches['riot_id_game_name'].isna()]

,match_id,puuid,riot_id_game_name,riot_id_tagline,kills,deaths,assists,win,team_position,gold_earned,gold_spent,team_id,champion_id,champion_name,time_played
43795,NA1_5569625662,tQi3fGiKUaoHd0e0a600rDxesKq34j4mQlqtjkoST1JSZH...,NaN,4897,3,7,7,True,BOTTOM,12628,11540,100,157,Yasuo,1722
230094,NA1_5570841322,tQi3fGiKUaoHd0e0a600rDxesKq34j4mQlqtjkoST1JSZH...,NaN,4897,6,2,5,True,BOTTOM,12162,11480,200,157,Yasuo,1461


In [144]:
# Смотрим уникальные значения в столбцах
display(f"Количество уникальных матчей: {df_players_matches['match_id'].nunique()}")
display(f"Количество уникальных PUUID игроков: {df_players_matches['puuid'].nunique()}")
display(f"Позиция в игре: {df_players_matches['team_position'].unique()}")
display(f"Количесвто уникальных имен используемых чемпионов: {df_players_matches['champion_name'].nunique()}")

'Количество уникальных матчей: 31959'

'Количество уникальных PUUID игроков: 76690'

"Позиция в игре: ['TOP' 'JUNGLE' 'MIDDLE' 'BOTTOM' 'UTILITY' nan]"

'Количесвто уникальных имен используемых чемпионов: 172'

In [145]:
# Убийства
df_players_matches['kills'].describe()

count    343524.000000
mean          5.802762
std           4.682759
min           0.000000
25%           2.000000
50%           5.000000
75%           8.000000
max          47.000000
Name: kills, dtype: float64

In [146]:
# Смерти
df_players_matches['deaths'].describe()

count    343524.000000
mean          5.822344
std           3.117557
min           0.000000
25%           4.000000
50%           6.000000
75%           8.000000
max          30.000000
Name: deaths, dtype: float64

In [147]:
# Ассисты
df_players_matches['assists'].describe()

count    343524.000000
mean          8.459784
std           6.361463
min           0.000000
25%           4.000000
50%           7.000000
75%          12.000000
max          69.000000
Name: assists, dtype: float64

In [148]:
# Заработанное золото за матч
df_players_matches['gold_earned'].describe()

count    343524.000000
mean      11810.361471
std        4366.778043
min         502.000000
25%        8826.000000
50%       11514.000000
75%       14501.000000
max       59883.000000
Name: gold_earned, dtype: float64

In [149]:
# Потраченное золото за матч
df_players_matches['gold_spent'].describe()

count    343524.000000
mean      11038.481527
std        4461.464253
min           0.000000
25%        8000.000000
50%       10700.000000
75%       13575.000000
max       77750.000000
Name: gold_spent, dtype: float64

### Добавление имени игрока в таблицу игроков

Внутриигровое имя игрока состоит из двух частей:

1. **riot_id_game_name** - никнейм — видимая часть из 3–16 символов.

2. **riot_id_tagline** - тег — комбинация из 3–5 букв или цифр, которая идет после знака #.

In [150]:
df_players_names = df_players_matches[['puuid','riot_id_game_name', 'riot_id_tagline']].drop_duplicates().sort_values(by='puuid')

In [151]:
#проверим на дубликаты по puuid игрока
display(f'Дубликатов по puuid игрока: {df_players_names.duplicated(subset = "puuid").sum()} строк')

'Дубликатов по puuid игрока: 784 строк'

В League of Legends одному puuid может соответствовать несколько разных связок Riot ID + Tagline. 

Причины: 
1. 1 раз в 90 дней разрешается менять ник или тэг в личном кабинете
2. Смена региона / Трансфер: Игрок переносит свой аккаунт на другой сервер, из-за чего его старый тег (например, #RU) автоматически меняется на новый (например, #EUW), но уникальный puuid остался прежним.

Т.к. в разных играх один и тоже игрок может называться по-разному, чтобы сформировать список игроков с именами будем выбирать из всех его имен самое последнее (актуальное).

In [152]:
# Выбираем игроков и id матчей
df_players_names = df_players_matches[['match_id', 'puuid','riot_id_game_name', 'riot_id_tagline']]

In [153]:
## Свяжем игроков и матчи
df = pd.merge(df_players_names, df_matches, on='match_id', how='left')

In [154]:
# Отсортируем по имени игрока и убыванию даты матча
df_sorted = df.sort_values(
    by=["puuid", "game_start_timestamp"], 
    ascending=[True, False]
)

In [155]:
# Удаляем дубликаты по puuid, оставляя только первую строку
df_latest_names = df_sorted.drop_duplicates(subset=["puuid"], keep="first")
display(f"Всего актуальных уникальных имен игроков: {df_latest_names.shape[0]}")

'Всего актуальных уникальных имен игроков: 76690'

In [156]:
# Оставим только нужные столбцы
df_latest_names = df_latest_names[['puuid','riot_id_game_name', 'riot_id_tagline']]

In [157]:
df_latest_names.head()

,puuid,riot_id_game_name,riot_id_tagline
151420,--4kaQKQ0_RaSFQ3Llxc3V-Zff81brUAhhYzeKZCKJqTGT...,Lapsivedenkeitin,EUW
225605,--B3eQJvOXzZJbdC2OrkwDBn_SxtKilQ9ZsmAHc5blZj97...,Solfato,000
239128,--BirFTljHXk01i50i0J6MaAdFC9Nj_XDlQ2-d2y5wJYPl...,Fraeya,Light
294004,--Hksuw1FZUFv1auEoiZ7FuSM_RKrLzfWSCUwiafB9YCyT...,Nasona,NA1
282543,--Nk6fhnIEavMg1ukXVOQv0UFuWPJgMujlibmMeEYA1b9q...,Sponjbawb,NA1


In [158]:
# Объединим таблицук игроков с именами, ЛЕВЫМ соединением
df_players = pd.merge(df_players, df_latest_names, on='puuid', how='left')

In [159]:
df_players.head()

,puuid,league_points,rank,wins,losses,veteran,inactive,fresh_blood,hot_streak,league_type,region,riot_id_game_name,riot_id_tagline
0,7qDgswqpOmHpVyAQO-085TwvVSp5kuojnI7ehOzZiBQX4J...,3814,I,787,643,True,False,False,False,CHALLENGER,EUW1,J1HUIV,000
1,D8z6JW2Ea_FfL6zRR98wcFmaMpK1KZRbYBGS3pIySYqNC0...,3722,I,263,201,True,False,False,False,CHALLENGER,EUW1,TheRoyalKanin,EUW
2,Sb7usV3fHduwHxZoXjwHdaVxF1T2LhaBHejvJdqOvRhZv_...,3598,I,749,628,True,False,False,False,CHALLENGER,EUW1,bro,han
3,ZIemTzkIPML6oScobsLKDZZCmGXzGA9yHbJAymoaqRNFop...,3594,I,386,314,True,False,False,False,CHALLENGER,EUW1,Nano,AFW
4,tlYvTzzoBLVbP6ZISTMT9V8mfibz4nNJVHkWbMSK5xlzF6...,3492,I,407,271,True,False,False,False,CHALLENGER,EUW1,Phanta,107


In [160]:
display(f'Получили имя {df_players['riot_id_game_name'].count()} игроков')

'Получили имя 17400 игроков'

### Справочник чемпионов

Сформируем справочник: код чемпиона и его имя, чтобы не тянуть лишнюю информацию в таблицу с игроками в матчах

In [161]:
df_champions = df_players_matches[['champion_id','champion_name']].drop_duplicates().sort_values(by='champion_id')

In [162]:
df_champions.head()

,champion_id,champion_name
90,1,Annie
461,2,Olaf
60,3,Galio
7,4,TwistedFate
162,5,XinZhao


In [163]:
display(f"Всего уникальных игровых чемпионов: {df_champions.shape[0]}")

'Всего уникальных игровых чемпионов: 172'

## Сохранение данных в БД Supabase

In [164]:
# Проверим типы данных перед загрузкой
display(df_players.dtypes) # Справочник игроков - puuid и имя
display(df_matches.dtypes) # Матчи
display(df_players_matches.dtypes) # Игроки в матчах
display(df_champions.dtypes) # Справочник чемпионов

puuid                object
league_points         int64
rank                 object
wins                  int64
losses                int64
veteran                bool
inactive               bool
fresh_blood            bool
hot_streak             bool
league_type          object
region               object
riot_id_game_name    object
riot_id_tagline      object
dtype: object

match_id                        object
game_name                       object
game_creation                    int64
game_duration                    int64
game_version                    object
game_mode                       object
end_of_game_result              object
game_start_timestamp             int64
game_end_timestamp               int64
game_type                       object
platform_id                     object
queue_id                         int64
game_creation_dt        datetime64[ns]
game_start_dt           datetime64[ns]
game_end_dt             datetime64[ns]
dtype: object

match_id             object
puuid                object
riot_id_game_name    object
riot_id_tagline      object
kills                 int64
deaths                int64
assists               int64
win                    bool
team_position        object
gold_earned           int64
gold_spent            int64
team_id               int64
champion_id           int64
champion_name        object
time_played           int64
dtype: object

champion_id       int64
champion_name    object
dtype: object

In [165]:
# Создаем список только тех колонок, которые нужны
# в игроках в матчах
required_columns_players = [
    'match_id',
    'puuid',
    'riot_id_game_name',
    'riot_id_tagline',
    'kills',
    'deaths',
    'assists',
    'win',
    'team_position',
    'gold_earned',
    'gold_spent',
    'team_id',
    'champion_id',
    'time_played'
    ]

### Подключение к БД

In [6]:
# Загрузка настроек подключения из .env
env_file = base_dir / "lol.env"
if env_file.is_file():
    # Загружаем 
    load_dotenv(env_file)
    display("Файл окружения загружен!")
else:
    display(f"❌ Ошибка: Файла нет в папке {base_dir}.")

'Файл окружения загружен!'

In [7]:
# Переменные подключения
USER = os.getenv("user")
PASSWORD = os.getenv("password")
HOST = os.getenv("host")
PORT = os.getenv("port")
DBNAME = os.getenv("dbname")

In [8]:
# Создание строки подключения SQLAlchemy 
DATABASE_URL = f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DBNAME}?sslmode=require"

In [9]:
# Create the SQLAlchemy engine
engine = create_engine(DATABASE_URL)

In [10]:
# Тест подключения
try:
    with engine.connect() as connection:
        display("Connection successful!")
except Exception as e:
    display(f"Failed to connect: {e}")

'Connection successful!'

### Таблица игроков  

In [171]:
# Удаляем предыдущие данные - связанные с таблицей представления 
try:
    with engine.connect() as conn:
        conn.execute(text("DROP TABLE IF EXISTS lol_players CASCADE;"))
        conn.commit()
        display('Успешно')
except Exception as e:
    display(f"❌ Произошла ошибка: {e}")    

'Успешно'

In [172]:
try:
    display(f"Начинаю загрузку имен игроков в Supabase - {df_players.shape[0]} записей ...")
    df_players.to_sql(
        name='lol_players',
        con=engine,
        if_exists='replace',
        index=False,
        chunksize=5000,
        method='multi'  # много строк для чанка
    )
    display("Игроки успешно записаны в Supabase в таблицу lol_players")
except Exception as e:
    display(f"❌ Произошла ошибка при загрузке: {e}")

'Начинаю загрузку имен игроков в Supabase - 21631 записей ...'

'Игроки успешно записаны в Supabase в таблицу lol_players'

### Таблица матчей 

In [67]:
# Создаем список только тех колонок, которые нужны
required_columns_matches = [
    'match_id',
    'game_creation_dt',
    'game_start_dt',
    'game_end_dt',
    'game_duration',
    'game_version',
    'game_mode',
    'platform_id',
    'queue_id'
    ]

In [68]:
# Удаляем предыдущие данные - связанные с таблицей представления 
try:
    with engine.connect() as conn:
        conn.execute(text("DROP TABLE IF EXISTS lol_matches CASCADE;"))
        conn.commit()
        display('Успешно')
except Exception as e:
    display(f"❌ Произошла ошибка: {e}")    

'Успешно'

In [69]:
try:
    display(f"Начинаю загрузку матчей в Supabase - {df_matches.shape[0]} записей ...")
    df_matches[required_columns_matches].to_sql(
        name='lol_matches',
        con=engine,
        if_exists='replace',
        index=False,
        chunksize=5000,
        method='multi'  # много строк для чанка
    )
    display("Матчи успешно записаны в Supabase в таблицу lol_matches")
except Exception as e:
    display(f"❌ Произошла ошибка при загрузке: {e}")

'Начинаю загрузку матчей в Supabase - 31959 записей ...'

'Матчи успешно записаны в Supabase в таблицу lol_matches'

### Таблица игроков в матчах 

In [70]:
# Удаляем предыдущие данные - связанные с таблицей представления 
try:
    with engine.connect() as conn:
        conn.execute(text("DROP TABLE IF EXISTS lol_players_matches CASCADE;"))
        conn.commit()
        display('Успешно')
except Exception as e:
    display(f"❌ Произошла ошибка: {e}")    

'Успешно'

In [71]:
try:
    display(f"Начинаю загрузку игроков в матчах в Supabase  - {df_players_matches.shape[0]} записей ...")
    df_players_matches[required_columns_players].to_sql(
        name='lol_players_matches',
        con=engine,
        if_exists='replace',
        index=False,
        chunksize=5000,
        method='multi'  # много строк для чанка
    )
    display("Игроки успешно записаны в Supabase в таблицу lol_players_matches")
except Exception as e:
    display(f"❌ Произошла ошибка при загрузке: {e}")

'Начинаю загрузку игроков в матчах в Supabase  - 343524 записей ...'

'Игроки успешно записаны в Supabase в таблицу lol_players_matches'

### Справочник чемпионов

In [72]:
# Удаляем предыдущие данные - связанные с таблицей представления 
try:
    with engine.connect() as conn:
        conn.execute(text("DROP TABLE IF EXISTS nsi_champions CASCADE;"))
        conn.commit()
        display('Успешно')
except Exception as e:
    display(f"❌ Произошла ошибка: {e}")    

'Успешно'

In [73]:
try:
    display(f"Начинаю загрузку справочника чемпионов в Supabase - {df_champions.shape[0]} записей ...")
    df_champions.to_sql(
        name='nsi_champions',
        con=engine,
        if_exists='replace',
        index=False,
        chunksize=5000,
        method='multi'  # много строк для чанка
    )
    display("Справочник чемпионов успешно записан в Supabase в таблицу nsi_champions")
except Exception as e:
    display(f"❌ Произошла ошибка при загрузке: {e}")

'Начинаю загрузку справочника чемпионов в Supabase - 172 записей ...'

'Справочник чемпионов успешно записан в Supabase в таблицу nsi_champions'

**Вывод:** Полученные по API данные предобработаны и успешно загружены в БД Supabase

## Создание витрин данных (Data Marts)

In [11]:
def create_supabase_view(loc_engine, sql_string):
    """ Создание представления в Supabase 
    sql_string - запрос SQL
    """
    try:
        # Открываем соединение и принудительно выполняем запрос внутри транзакции
        with loc_engine.begin() as connection:
            connection.execute(text(sql_string))
            display("🎉 Представление успешно создано в Supabase через SQLAlchemy!")
    except Exception as e:
        display(f"❌ Ошибка при создании представления: {e}")    
    return    

### Матчи

In [113]:
# -- Индикаторы по регионам: количество матчей, средняя длительность 
create_view_query = """
CREATE OR REPLACE VIEW public.v_matches_indicators_region AS
SELECT 
	platform_id AS region,
	COUNT(DISTINCT match_id) as total_matches,
	ROUND(AVG(game_duration) / 60.0, 2) AS avg_duration_minutes
FROM public.lol_matches
GROUP by platform_id
UNION ALL
SELECT 
    'Все регионы' AS region,
	COUNT(DISTINCT match_id) as total_matches,
	ROUND(AVG(game_duration) / 60.0, 2) AS avg_duration_minutes
FROM public.lol_matches; 
"""

In [114]:
create_supabase_view(engine, create_view_query)

'🎉 Представление успешно создано в Supabase через SQLAlchemy!'

In [103]:
#-- Среднее Количество матчей в день
create_view_query = """
CREATE OR REPLACE VIEW public.v_avg_games_day AS
with count_matches_day as(
SELECT 
	platform_id AS region,
    game_start_dt::date AS match_date, -- Отрезаем время, оставляем только ГГГГ-ММ-ДД
    COUNT(DISTINCT match_id) AS games_count
FROM public.lol_matches
WHERE game_start_dt IS NOT NULL
GROUP BY platform_id, game_start_dt::date
UNION ALL
SELECT 
    'Все регионы' AS region,
    game_start_dt::date AS match_date,
    COUNT(DISTINCT match_id) AS games_count
FROM public.lol_matches
WHERE game_start_dt IS NOT NULL
GROUP BY game_start_dt::date
)
select	region,
		round(avg(games_count),2) as avg_games_day
from count_matches_day
group by region;		
"""

In [104]:
create_supabase_view(engine, create_view_query)

'🎉 Представление успешно создано в Supabase через SQLAlchemy!'

In [109]:
# Среднее количество уникальных игроков по регионам в день за период (месяц)
create_view_query = """
CREATE OR REPLACE VIEW public.v_avg_players_day AS
with count_players_days as (
SELECT 
    split_part(p.match_id, '_', 1) AS region,
    m.game_start_dt::date AS match_date, -- Берём дату из таблицы матчей
    COUNT(distinct p.puuid) AS players_count -- Считаем общее количество участников
FROM public.lol_players_matches AS p
JOIN public.lol_matches AS m ON m.match_id = p.match_id
WHERE m.game_start_dt between '01-may-2026' and '31-may-2026'
GROUP BY split_part(p.match_id, '_', 1), m.game_start_dt::date
UNION ALL
SELECT 
    'Все регионы' AS region,
    m.game_start_dt::date AS match_date,
    COUNT(distinct p.puuid) AS players_count
FROM public.lol_players_matches AS p
JOIN public.lol_matches AS m ON m.match_id = p.match_id
WHERE m.game_start_dt between '01-may-2026' and '31-may-2026'
GROUP BY m.game_start_dt::date
)
select 
	region,
	avg(players_count)::int as avg_players_day
from count_players_days
group by region;
"""

In [110]:
create_supabase_view(engine, create_view_query)

'🎉 Представление успешно создано в Supabase через SQLAlchemy!'

In [115]:
#-- Количество матчей в день - линейный график
create_view_query = """
CREATE OR REPLACE VIEW public.v_matches_date AS
SELECT 
	platform_id AS region,
    game_start_dt::date AS match_date, -- Отрезаем время, оставляем только ГГГГ-ММ-ДД
    COUNT(DISTINCT match_id) AS games_count
FROM public.lol_matches
WHERE game_start_dt IS NOT NULL
GROUP BY platform_id, game_start_dt::date
UNION ALL
SELECT 
    'Все регионы' AS region,
    game_start_dt::date AS match_date,
    COUNT(DISTINCT match_id) AS games_count
FROM public.lol_matches
WHERE game_start_dt IS NOT NULL
GROUP BY game_start_dt::date;
"""

In [116]:
create_supabase_view(engine, create_view_query)

'🎉 Представление успешно создано в Supabase через SQLAlchemy!'

In [117]:
#--- количество уникальных игроков по регионам в день по дням
create_view_query = """
CREATE OR REPLACE VIEW public.v_players_date AS
SELECT 
    split_part(p.match_id, '_', 1) AS region,
    m.game_start_dt::date AS match_date, -- Берём дату из таблицы матчей
    COUNT(distinct p.puuid) AS players_count -- Считаем общее количество участников
FROM public.lol_players_matches AS p
JOIN public.lol_matches AS m ON m.match_id = p.match_id
WHERE m.game_start_dt between '01-may-2026' and '31-may-2026'
GROUP BY split_part(p.match_id, '_', 1), m.game_start_dt::date
UNION ALL
SELECT 
    'Все регионы' AS region,
    m.game_start_dt::date AS match_date,
    COUNT(distinct p.puuid) AS players_count
FROM public.lol_players_matches AS p
JOIN public.lol_matches AS m ON m.match_id = p.match_id
WHERE m.game_start_dt between '01-may-2026' and '31-may-2026'
GROUP BY m.game_start_dt::date
order by match_date;
"""

In [118]:
create_supabase_view(engine, create_view_query)

'🎉 Представление успешно создано в Supabase через SQLAlchemy!'

In [125]:
#-- Для диаграммы размаха длительности матча в зависимости от версии игры
#-- Исключаем ремейки (игры короче 5 минут) для корректности аналитики
create_view_query = """
CREATE OR REPLACE VIEW public.v_boxplot_duration_patch AS
SELECT 
	platform_id as region,    
	game_version,
    COUNT(*) as total_matches,
    round(MIN(game_duration / 60.0)::numeric,2) AS min_duration,
    round(PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY game_duration / 60.0) :: numeric,2) AS q1,
    round(PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY game_duration / 60.0) :: numeric,2) AS median,
    round(PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY game_duration / 60.0) :: numeric,2) AS q3,
    round(MAX(game_duration / 60.0)::numeric,2) AS max_duration
FROM public.lol_matches
WHERE game_duration > 300 
GROUP BY region, game_version
union all 
SELECT 
	'Все регионы' AS region,    
	game_version,
    COUNT(*) as total_matches,
    round(MIN(game_duration / 60.0)::numeric,2) AS min_duration,
    round(PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY game_duration / 60.0) :: numeric,2) AS q1,
    round(PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY game_duration / 60.0) :: numeric,2) AS median,
    round(PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY game_duration / 60.0) :: numeric,2) AS q3,
    round(MAX(game_duration / 60.0)::numeric,2) AS max_duration
FROM public.lol_matches
WHERE game_duration > 300 
GROUP BY region, game_version
ORDER BY game_version desc;
"""

In [126]:
create_supabase_view(engine, create_view_query)

'🎉 Представление успешно создано в Supabase через SQLAlchemy!'

In [139]:
#Корреляция между длительностью матча и исходом
#-- Исключаем ремейки - специальная функция в League of Legends, которая позволяет игрокам досрочно завершить матч, 
# если один из союзников вылетел из игры 
#-- По регионам
create_view_query = """
CREATE OR REPLACE VIEW public.v_corr_wins_duration_region AS
SELECT 
    m.platform_id AS region, 
    pm.team_id,
    FLOOR(m.game_duration / 60.0) AS match_minute, 
    COUNT(*) AS total_matches,
    SUM(CASE WHEN pm.win = true THEN 1 ELSE 0 END) AS wins_count,
    ROUND((SUM(CASE WHEN pm.win = true THEN 1 ELSE 0 END) * 100.0) / COUNT(*),2) AS win_rate
FROM public.lol_players_matches AS pm
JOIN public.lol_matches AS m ON m.match_id = pm.match_id
WHERE m.game_duration > 300 
GROUP BY m.platform_id, pm.team_id, FLOOR(m.game_duration / 60.0);
"""

In [140]:
create_supabase_view(engine, create_view_query)

'🎉 Представление успешно создано в Supabase через SQLAlchemy!'

In [137]:
#Корреляция между длительностью матча и исходом
#-- По ВСЕМ регионам
create_view_query = """
CREATE OR REPLACE VIEW public.v_corr_wins_duration AS
SELECT 
    'Все регионы' AS region,    
    team_id,
    FLOOR(game_duration / 60.0) AS match_minute,
    COUNT(*) AS total_matches,
    SUM(CASE WHEN win = true THEN 1 ELSE 0 END) AS wins_count,
    ROUND((SUM(CASE WHEN win = true THEN 1 ELSE 0 END) * 100.0) / COUNT(*),2) AS win_rate
FROM public.lol_players_matches AS pm
JOIN public.lol_matches AS m ON m.match_id = pm.match_id
WHERE m.game_duration > 300 -- Исключаем ремейки короче 5 минут для точности тренда
GROUP BY team_id, FLOOR(game_duration / 60.0);
"""

In [138]:
create_supabase_view(engine, create_view_query)

'🎉 Представление успешно создано в Supabase через SQLAlchemy!'

In [145]:
#-- Процент побед по командам - для кольцевой диаграммы
#-- Исключаем ремейки
create_view_query = """
CREATE OR REPLACE VIEW public.v_side_winrate AS
SELECT 
    m.platform_id AS region,
    pm.team_id, 
    COUNT(*) AS total_games,
    SUM(CASE WHEN pm.win = true THEN 1 ELSE 0 END) AS wins_count
from public.lol_players_matches AS pm
JOIN public.lol_matches AS m ON m.match_id = pm.match_id
where m.game_duration > 300 
GROUP BY m.platform_id, pm.team_id
union ALL 
SELECT 
    'Все регионы' AS region,    
    pm.team_id, 
    COUNT(*) AS total_games,
    SUM(CASE WHEN pm.win = true THEN 1 ELSE 0 END) AS wins_count
from public.lol_players_matches AS pm
JOIN public.lol_matches AS m ON m.match_id = pm.match_id
where m.game_duration > 300 
GROUP BY pm.team_id;
"""

In [146]:
create_supabase_view(engine, create_view_query)

'🎉 Представление успешно создано в Supabase через SQLAlchemy!'

In [147]:
# Индекс «Кровавости» (Kills Per Minute)
#-- Исключаем ремейки
create_view_query = """
CREATE OR REPLACE VIEW public.v_kpm_by_version AS
with total_kills_match as(     -- Считаем убийства в каждом матче
select	match_id,
		sum(kills) as total_kills
from public.lol_players_matches
group by match_id
),
match_data as (               -- Присоединяем длительность и регион
select	m.platform_id,
		k.match_id,
		m.game_version, 
		m.game_duration,
		k.total_kills
from total_kills_match as k
join public.lol_matches as m on m.match_id=k.match_id
),
kpi_region as (
select	platform_id as region,
		game_version,
		-- Считаем суммарное количество матчей 
    	COUNT(*) AS total_matches,
		ROUND(AVG(total_kills / (game_duration / 60.0)), 2) AS kpm
from match_data
where  game_duration > 300 -- Исключаем технические ремейки
GROUP BY platform_id, game_version
),
kpi_all_region as(
select	'Все регионы' as region,
		game_version,
		-- Считаем суммарное количество матчей 
    	COUNT(*) AS total_matches,
		ROUND(AVG(total_kills / (game_duration / 60.0)), 2) AS kpm
from match_data
where  game_duration > 300 -- Исключаем технические ремейки
GROUP BY game_version
)
select * from kpi_all_region
union all
select * from kpi_region
order by game_version, region;
"""

In [148]:
create_supabase_view(engine, create_view_query)

'🎉 Представление успешно создано в Supabase через SQLAlchemy!'

### Игроки

In [173]:
#--- KPI для индикаторов при выборе лиги
#-- 1. Всего игроков в лиге
#-- 2. Средний винрейт игроков
#-- 3. Среднее количество матчей на человека
#-- 4. Рекорд рейтинга (Максимальный LP)
create_view_query = """
CREATE OR REPLACE VIEW public.v_kpi_league  AS
with kpi_region as (
select	region,
	    league_type,
    	COUNT(*) AS total_players,
	    ROUND(AVG((wins * 100.0) / NULLIF(wins + losses, 0)), 1) AS avg_winrate,
		ROUND(AVG(wins + losses), 0) AS avg_matches,
    	MAX(league_points) AS max_lp
FROM public.lol_players
GROUP BY region, league_type
),
kpi_all_region as (
select	'Все регионы' as region,
	    league_type,
    	COUNT(*) AS total_players,
	    ROUND(AVG((wins * 100.0) / NULLIF(wins + losses, 0)), 1) AS avg_winrate,
		ROUND(AVG(wins + losses), 0) AS avg_matches,
    	MAX(league_points) AS max_lp
FROM public.lol_players
GROUP BY league_type
)
select * from kpi_region
union all
select * from kpi_all_region
order by region, league_type;
"""

In [174]:
create_supabase_view(engine, create_view_query)

'🎉 Представление успешно создано в Supabase через SQLAlchemy!'

In [16]:
#---- Средний КДА - спидометр
#--- Объединим игроков в матчах со списком игроков
create_view_query = """
CREATE OR REPLACE VIEW public.v_avg_kda  AS
with player_kda as (
select 
	p.region,
	p.league_type,
	p.puuid,
	pm.match_id,
	pm.kills,
	pm.assists,
	pm.deaths
from public.lol_players p
left join public.lol_players_matches pm on pm.puuid=p.puuid
),
avg_league_region_kda as (
select	region,
    	league_type,
    	-- Рассчитываем средневзвешенный KDA лиги по конкретному региону
    	ROUND((SUM(kills) + SUM(assists))::numeric / NULLIF(SUM(deaths), 0), 2) AS avg_kda
from player_kda
GROUP BY region, league_type
),
avg_league_all_region_kda as (
select 'Все регионы' as region,
    	league_type,
    	-- считаем суммарный мировой KDA напрямую из логов матчей
    	ROUND((SUM(kills) + SUM(assists))::numeric / NULLIF(SUM(deaths), 0), 2) AS avg_kda
from player_kda
GROUP BY league_type
)
select * from avg_league_region_kda
union all
select * from avg_league_all_region_kda
order by region, league_type;
"""

In [17]:
create_supabase_view(engine, create_view_query)

'🎉 Представление успешно создано в Supabase через SQLAlchemy!'

In [14]:
#---- Индекс агрессии K/A Ratio - спидометр
create_view_query = """
CREATE OR REPLACE VIEW public.v_avg_ka  AS
with player_ka as (
select 
	p.region,
	p.league_type,
	p.puuid,
	pm.match_id,
	pm.kills,
	pm.assists
from public.lol_players p
left join public.lol_players_matches pm on pm.puuid=p.puuid
),
avg_league_region_ka as (
select	region,
    	league_type,
    	-- Рассчитываем средневзвешенный KA лиги по региону
    	ROUND(SUM(kills)::numeric / NULLIF(SUM(assists) + SUM(kills), 0), 2) as avg_ka 
from player_ka
GROUP BY region, league_type
),
avg_league_all_region_ka as (
select 'Все регионы' as region,
    	league_type,
    	-- считаем глобальную сумму по всем регионам вместе, а не среднее от средних
    	ROUND(SUM(kills)::numeric / NULLIF(SUM(assists) + SUM(kills), 0), 2) as avg_ka
from player_ka
GROUP BY league_type
)
select * from avg_league_region_ka
union all
select * from avg_league_all_region_ka
order by region, league_type;
"""

In [15]:
create_supabase_view(engine, create_view_query)

'🎉 Представление успешно создано в Supabase через SQLAlchemy!'

In [175]:
#--- Диаграмма распределения LP игроков
create_view_query = """
CREATE OR REPLACE VIEW public.v_lp_distribution  AS
with LP_region as (
SELECT 
    region,
    league_type,
    -- Округляем LP до ближайшего десятка, чтобы создать бакеты для гистограммы
    FLOOR(league_points / 10.0) * 10 AS lp_bucket,
    -- Считаем количество игроков в каждой корзине
    COUNT(*) AS players_count
FROM public.lol_players
WHERE league_points IS NOT NULL AND league_points >= 0
GROUP BY region, league_type, FLOOR(league_points / 10.0) * 10
),
LP_all_region as (
SELECT 
    'Все регионы' as region,
    league_type,
    -- Округляем LP до ближайшего десятка, чтобы создать бакеты для гистограммы
    FLOOR(league_points / 10.0) * 10 AS lp_bucket,
    -- Считаем количество игроков в каждой корзине
    COUNT(*) AS players_count
FROM public.lol_players
WHERE league_points IS NOT NULL AND league_points >= 0
GROUP BY region, league_type, FLOOR(league_points / 10.0) * 10
)
select * from LP_region
union all
select * from LP_all_region
order by region, league_type, lp_bucket;
"""

In [176]:
create_supabase_view(engine, create_view_query)

'🎉 Представление успешно создано в Supabase через SQLAlchemy!'

In [18]:
#--- Показатели по игроку
create_view_query = """
CREATE OR REPLACE VIEW public.v_player_kpi  AS
with players_matches_kpi as (
select	puuid,
		match_id,
		win::int ,
		1 - win::int as losses,
		kills,
		deaths,
		assists,
		gold_earned,
		gold_spent,
		team_id,
		CASE 
    		WHEN deaths = 0 THEN round((kills + assists)::numeric / 1.0, 2)
		    ELSE round((kills + assists)::numeric / deaths, 2)
		END AS kda,
		ROUND(kills::numeric / NULLIF(assists + kills, 0), 2) as ka
from public.lol_players_matches 
)
--- присоединяем имя, группируем по игроку и считаем среднее по игроку + WinRate
select  p.puuid,
		p.region,
		p.league_type,
		COALESCE(p.riot_id_game_name || '#' || p.riot_id_game_name, p.puuid) as player_name,
		sum(p.league_points) as lp, 
		sum(p.wins) as wins,
		sum(p.losses) as losses,
		count(pm.match_id) as total_games,
		sum(pm.win) as wins_current,
		sum(pm.losses) as losses_current,
		sum(pm.kills) as kills,
		sum(pm.deaths) as deaths,
		sum(pm.assists) as assists,
		sum(pm.gold_earned) as gold_earned,
		sum(pm.gold_spent) as gold_spent,
		ROUND(sum((p.wins) * 100.0)::numeric / NULLIF(sum(p.wins) + sum(p.losses), 1),2) AS winrate,
		round(avg(pm.kda),2) as avg_kda,
		round(avg(pm.ka),2) as avg_ka
from players_matches_kpi as pm 
join  public.lol_players as p on p.puuid=pm.puuid 
group by	p.region,
			p.league_type,
			COALESCE(p.riot_id_game_name || '#' || p.riot_id_game_name, p.puuid),
			p.puuid;
"""

In [19]:
create_supabase_view(engine, create_view_query)

'🎉 Представление успешно создано в Supabase через SQLAlchemy!'

In [16]:
#Предпочитаемый чемпион, позиция и команда
create_view_query = """
CREATE OR REPLACE VIEW public.v_player_favorites  AS
WITH main_players AS (
    -- Получаем список уникальных игроков (чтобы ничего не потерять)
    SELECT DISTINCT puuid 
    FROM public.lol_players_matches
),
fav_champions AS (
    -- Ищем самого частого чемпиона для каждого игрока
    SELECT puuid, champion_id AS favorite_champion
    FROM (select	puuid,
    		 	 	champion_id,
               		ROW_NUMBER() OVER(PARTITION BY puuid ORDER BY COUNT(*) DESC) as rn
        	FROM public.lol_players_matches
        	WHERE champion_id IS NOT NULL
        	GROUP BY puuid, champion_id
    	) t
    WHERE rn = 1
),
fav_positions AS (
    -- Ищем самую частую позицию (роль) для каждого игрока
    SELECT puuid, team_position AS favorite_position
    FROM (SELECT puuid,
    			 team_position,
                 ROW_NUMBER() OVER(PARTITION BY puuid ORDER BY COUNT(*) DESC) as rn
          FROM public.lol_players_matches
          WHERE team_position IS NOT NULL AND team_position <> ''
          GROUP BY puuid, team_position
    ) t
    WHERE rn = 1
),
fav_teams AS (
    -- Ищем самую частую сторону (team_id: Синие/Красные)
    select	puuid,
    		team_id AS favorite_team_side
    FROM ( SELECT puuid, team_id,
           ROW_NUMBER() OVER(PARTITION BY puuid ORDER BY COUNT(*) DESC) as rn
       	   FROM public.lol_players_matches
        WHERE team_id IS NOT NULL
        GROUP BY puuid, team_id
    ) t
    WHERE rn = 1
)
-- Собираем всё воедино в одну плоскую таблицу
SELECT 
    p.puuid,
    c.favorite_champion,
    coalesce(nsi.champion_name,'Нет предпочтения') as champion_name,
    coalesce(pos.favorite_position,'Нет предпочтения') as team_position,
    CASE
	    when t.favorite_team_side = 100 then 'Синяя'
	    when t.favorite_team_side = 200 then 'Красная'
    ELSE 'Нет предпочтения'
    END as team
FROM main_players p
LEFT JOIN fav_champions c ON p.puuid = c.puuid
LEFT JOIN fav_positions pos ON p.puuid = pos.puuid
LEFT JOIN fav_teams t ON p.puuid = t.puuid
inner join public.nsi_champions as nsi on c.favorite_champion = nsi.champion_id;
"""

In [17]:
create_supabase_view(engine, create_view_query)

'🎉 Представление успешно создано в Supabase через SQLAlchemy!'

### Чемпионы

In [88]:
#--- Общее количество чемпионов по всем регионам - для индикатора
create_view_query = """
CREATE OR REPLACE VIEW public.v_total_champions_count AS
SELECT COUNT(DISTINCT p.champion_id) AS total_champions_count
from public.lol_players_matches as p;
"""

In [89]:
create_supabase_view(engine, create_view_query)

'🎉 Представление успешно создано в Supabase через SQLAlchemy!'

In [90]:
#--- Общее количество чемпионов по регионам - для индикатора
create_view_query = """
CREATE OR REPLACE VIEW public.v_total_champions_count_region AS
select	split_part(p.match_id, '_', 1) as region,
		COUNT(DISTINCT p.champion_id) AS total_champions_count
from public.lol_players_matches as p
group by split_part(p.match_id, '_', 1);
"""

In [91]:
create_supabase_view(engine, create_view_query)

'🎉 Представление успешно создано в Supabase через SQLAlchemy!'

In [95]:
#--- Средние показатели KDA, kills, gold по всем регионам
create_view_query = """
CREATE OR REPLACE VIEW public.v_champions_indicators AS
SELECT ROUND(AVG((kills + assists) / NULLIF(deaths, 0)::numeric), 2) AS total_avg_kda,
       ROUND(AVG(kills)::numeric, 1) AS total_avg_kills,
       ROUND(AVG(gold_earned)::numeric, 0) AS total_avg_gold
FROM public.lol_players_matches;
"""

In [96]:
create_supabase_view(engine, create_view_query)

'🎉 Представление успешно создано в Supabase через SQLAlchemy!'

In [97]:
#--- Средние показатели KDA, kills, gold по всем регионам
create_view_query = """
CREATE OR REPLACE VIEW public.v_champions_indicators_region AS
SELECT 	split_part(match_id, '_', 1) as region,
		ROUND(AVG((kills + assists) / NULLIF(deaths, 0)::numeric), 2) AS total_avg_kda,
	    ROUND(AVG(kills)::numeric, 1) AS total_avg_kills,
    	ROUND(AVG(gold_earned)::numeric, 0) AS total_avg_gold
FROM public.lol_players_matches
group by split_part(match_id, '_', 1);
"""

In [98]:
create_supabase_view(engine, create_view_query)

'🎉 Представление успешно создано в Supabase через SQLAlchemy!'

In [83]:
# Топ-15 чемпионов по Win Rate по всем регионам
create_view_query = """
CREATE OR REPLACE VIEW public.v_top_champions_winrate AS
WITH top_champions_winrate AS( 
	SELECT	p.champion_id,
			nc.champion_name,
    		ROUND(SUM(p.win::int)::numeric/(count(p.win))*100,2) AS winrate,
    		ROW_NUMBER() OVER (ORDER BY (SUM(p.win::int)::numeric/count(p.win)) desc) AS champion_rank
    FROM public.lol_players_matches as p
    join public.nsi_champions as nc on nc.champion_id = p.champion_id 
	group by p.champion_id, nc.champion_name
    ),
filtered_top_champions AS (
    SELECT * 
    FROM top_champions_winrate
    WHERE champion_rank <= 15
)    
SELECT * 
FROM filtered_top_champions;
"""

In [84]:
create_supabase_view(engine, create_view_query)

'🎉 Представление успешно создано в Supabase через SQLAlchemy!'

In [79]:
# Топ-15 чемпионов по Win Rate по региону 
create_view_query = """
CREATE OR REPLACE VIEW public.v_top_champions_winrate_regions AS
WITH top_champions_winrate AS( 
	SELECT	split_part(p.match_id, '_', 1) as region,
			p.champion_id,
			nc.champion_name,
    		ROUND(SUM(p.win::int)::numeric/(count(p.win))*100,2) AS winrate,
    		ROW_NUMBER() OVER (PARTITION BY split_part(p.match_id, '_', 1) ORDER BY (SUM(p.win::int)::numeric/count(p.win)) desc) AS champion_rank
    FROM public.lol_players_matches as p
    join public.nsi_champions as nc on nc.champion_id = p.champion_id 
	group by region, p.champion_id, nc.champion_name
    ),
filtered_top_champions AS (
    SELECT * 
    FROM top_champions_winrate
    WHERE champion_rank <= 15
)    
SELECT * 
FROM filtered_top_champions;
"""

In [80]:
create_supabase_view(engine, create_view_query)

'🎉 Представление успешно создано в Supabase через SQLAlchemy!'

In [86]:
# Топ-15 чемпионов по Популярности 
create_view_query = """
CREATE OR REPLACE VIEW public.v_top_champions_pickrate AS
WITH total_match_count as(
	SELECT COUNT(DISTINCT match_id) 
	FROM public.lol_matches),
top_champions_pickrate AS( 
	SELECT	p.champion_id,
			nc.champion_name,
    		ROUND(count(*)::numeric/(select* from total_match_count)*100,2) AS pickrate,
    		ROW_NUMBER() OVER (ORDER BY count(*)::numeric/(select* from total_match_count) desc) AS champion_rank
    FROM public.lol_players_matches as p
    join public.nsi_champions as nc on nc.champion_id = p.champion_id 
	group by p.champion_id, nc.champion_name
    )
SELECT * 
FROM top_champions_pickrate
where champion_rank <= 15
order by champion_rank;
"""

In [87]:
create_supabase_view(engine, create_view_query)

'🎉 Представление успешно создано в Supabase через SQLAlchemy!'

In [81]:
# Топ-15 чемпионов по Популярности по регионам
create_view_query = """
CREATE OR REPLACE VIEW public.v_top_champions_pickrate_regions AS
WITH total_match_count as(
	SELECT	split_part(match_id, '_', 1) as region,
			COUNT(DISTINCT match_id) as matches_count
	FROM public.lol_matches
	group by split_part(match_id, '_', 1)
	),
champion_counts AS (
    SELECT	
        split_part(p.match_id, '_', 1) AS region,
        p.champion_id,
        nc.champion_name,
        COUNT(*) AS picks_count
    FROM public.lol_players_matches AS p
    JOIN public.nsi_champions AS nc ON nc.champion_id = p.champion_id 
    GROUP BY split_part(p.match_id, '_', 1), p.champion_id, nc.champion_name
),
top_champions_pickrate AS (
    SELECT 
        c.region,
        c.champion_id,
        c.champion_name,
        c.picks_count,
        ROUND((c.picks_count::numeric / t.matches_count) * 100, 2) AS pickrate,
        ROW_NUMBER() OVER (PARTITION BY c.region  ORDER BY c.picks_count DESC) AS champion_rank
    FROM champion_counts AS c
    JOIN total_match_count AS t ON c.region = t.region
),
filtered_top_champions AS (
    SELECT * 
    FROM top_champions_pickrate
    WHERE champion_rank <= 15
)    
SELECT * 
FROM filtered_top_champions;
"""

In [82]:
create_supabase_view(engine, create_view_query)

'🎉 Представление успешно создано в Supabase через SQLAlchemy!'

In [18]:
# --- Витрина для пузырьковой диаграммы - зависимость побед от популярности (Win Rate Pick от Rate) для всех игровых чемпионов
# -- Для выбранного региона
create_view_query = """
CREATE OR REPLACE VIEW public.v_champions_kpi_region AS
WITH total_match_count AS (
    SELECT	
        split_part(match_id, '_', 1) AS region,
        COUNT(DISTINCT match_id) AS matches_count
    FROM public.lol_matches
    GROUP BY split_part(match_id, '_', 1)
),
champion_stats AS (
    SELECT	
        split_part(p.match_id, '_', 1) AS region,
        p.champion_id,
        nc.champion_name,
        COUNT(*) AS picks_count,
        COUNT(CASE WHEN p.win = true THEN 1 END) AS wins_count
    FROM public.lol_players_matches AS p
    JOIN public.nsi_champions AS nc ON nc.champion_id = p.champion_id 
    GROUP BY split_part(p.match_id, '_', 1), p.champion_id, nc.champion_name
),
top_champions_metrics AS (
    SELECT 
        c.region,
        c.champion_id,
        c.champion_name,
        c.picks_count,
        c.wins_count,
        ROUND((c.picks_count::numeric / t.matches_count) * 100, 2) AS pickrate,
        ROUND((c.wins_count::numeric / c.picks_count) * 100, 2) AS winrate
        FROM champion_stats AS c
    JOIN total_match_count AS t ON c.region = t.region
)
SELECT 
    region,
    champion_id,
    champion_name,
    picks_count,
    wins_count,
    pickrate,
    winrate
FROM top_champions_metrics;
"""

In [19]:
create_supabase_view(engine, create_view_query)

'🎉 Представление успешно создано в Supabase через SQLAlchemy!'

In [22]:
# --- Витрина для пузырьковой диаграммы - зависимость побед от популярности (Win Rate Pick от Rate) для всех игровых чемпионов
# -- Для всех регионов
create_view_query = """
CREATE OR REPLACE VIEW public.v_champions_kpi AS
with champion_stats AS (
    SELECT	
        p.champion_id,
        nc.champion_name,
        COUNT(*) AS picks_count,
        COUNT(CASE WHEN p.win = true THEN 1 END) AS wins_count
    FROM public.lol_players_matches AS p
    JOIN public.nsi_champions AS nc ON nc.champion_id = p.champion_id 
    GROUP BY p.champion_id, nc.champion_name
),
top_champions_metrics AS (
    SELECT 
        'Все регионы' as region,
        c.champion_id,
        c.champion_name,
        c.picks_count,
        c.wins_count,
        ROUND((c.picks_count::numeric / (SELECT COUNT(DISTINCT match_id) FROM public.lol_matches)) *100, 2)  AS pickrate,
        ROUND((c.wins_count::numeric / c.picks_count) * 100, 2) AS winrate
        FROM champion_stats AS c
    )
SELECT 
    region,
    champion_id,
    champion_name,
    picks_count,
    wins_count,
    pickrate,
    winrate
FROM top_champions_metrics;
"""

In [23]:
create_supabase_view(engine, create_view_query)

'🎉 Представление успешно создано в Supabase через SQLAlchemy!'

In [20]:
# Витрина для scatter-plot зависимостей средних убийств от смертей по чемпионам
create_view_query = """
CREATE OR REPLACE VIEW public.mv_champion_kill_death_stats AS
SELECT 
    split_part(p.match_id, '_', 1) AS region,
    nc.champion_name,
    COUNT(*) AS total_matches,
    ROUND(AVG(p.kills)::numeric, 2) AS avg_kills,
    ROUND(AVG(p.deaths)::numeric, 2) AS avg_deaths
FROM public.lol_players_matches AS p
JOIN public.nsi_champions AS nc ON nc.champion_id = p.champion_id
GROUP BY split_part(p.match_id, '_', 1), nc.champion_name
UNION ALL
SELECT 
    'Все регионы' AS region,
    nc.champion_name,
    COUNT(*) AS total_matches,
    ROUND(AVG(p.kills)::numeric, 2) AS avg_kills,
    ROUND(AVG(p.deaths)::numeric, 2) AS avg_deaths
FROM public.lol_players_matches AS p
JOIN public.nsi_champions AS nc ON nc.champion_id = p.champion_id
GROUP BY nc.champion_name;
"""

In [21]:
create_supabase_view(engine, create_view_query)

'🎉 Представление успешно создано в Supabase через SQLAlchemy!'

In [31]:
# -- Витрина для анализа гибкости позиций игровых чемпионов
create_view_query = """
CREATE OR REPLACE VIEW public.v_champion_positions AS
SELECT 
    split_part(p.match_id, '_', 1) AS region,
    p.champion_id,
    nc.champion_name,
    p.team_position,
    COUNT(*) AS games_on_position
FROM public.lol_players_matches AS p
JOIN public.nsi_champions AS nc ON nc.champion_id = p.champion_id
WHERE p.team_position IS NOT NULL AND p.team_position != ''
GROUP BY split_part(p.match_id, '_', 1), p.champion_id, nc.champion_name, p.team_position
UNION ALL
SELECT 
    'Все регионы' AS region,
    p.champion_id,
    nc.champion_name,
    p.team_position,
    COUNT(*) AS games_on_position
FROM public.lol_players_matches AS p
JOIN public.nsi_champions AS nc ON nc.champion_id = p.champion_id
WHERE p.team_position IS NOT NULL AND p.team_position != ''
GROUP BY p.champion_id, nc.champion_name, p.team_position;
"""

In [32]:
create_supabase_view(engine, create_view_query)

'🎉 Представление успешно создано в Supabase через SQLAlchemy!'

In [25]:
#--- Кривая силы чемпиона от времени игры для выбранного региона
#-- Отсекаем редкие матчи, чтобы избежать статистических аномалий (< 5)
create_view_query = """
CREATE OR REPLACE VIEW public.v_champion_power_curve_region AS
WITH match_intervals AS (
    SELECT 
        split_part(m.match_id, '_', 1) AS region,
        p.champion_id,
        nc.champion_name,
        p.win,
        CASE 
            WHEN m.game_duration / 60 < 20 THEN '1. <20 мин (FF/Сдались)'
            WHEN m.game_duration / 60 >= 20 AND m.game_duration / 60 < 25 THEN '2. 20-25 мин (Ранняя)'
            WHEN m.game_duration / 60 >= 25 AND m.game_duration / 60 < 30 THEN '3. 25-30 мин (Мид-гейм)'
            WHEN m.game_duration / 60 >= 30 AND m.game_duration / 60 < 35 THEN '4. 30-35 мин (Лейт-гейм)'
            ELSE '5. 35+ мин (Глубокий лейт)'
        END AS game_duration_interval
    FROM public.lol_players_matches AS p
    JOIN public.lol_matches AS m ON m.match_id = p.match_id
    JOIN public.nsi_champions AS nc ON nc.champion_id = p.champion_id
),
aggregated_stats AS (
    SELECT 
        region,
        champion_id,
        champion_name,
        game_duration_interval,
        COUNT(*) AS total_games,
        COUNT(CASE WHEN win = true THEN 1 END) AS wins_count
    FROM match_intervals
    GROUP BY region, champion_id, champion_name, game_duration_interval
)
SELECT 
    region,
    champion_id,
    champion_name,
    game_duration_interval,
    total_games,
    wins_count,
    ROUND((wins_count::numeric / total_games) * 100, 2) AS winrate
FROM aggregated_stats
WHERE total_games >= 5; 
"""

In [26]:
create_supabase_view(engine, create_view_query)

'🎉 Представление успешно создано в Supabase через SQLAlchemy!'

In [29]:
#--- Кривая силы чемпиона от времени игры для ВСЕХ регионов
#-- Отсекаем редкие матчи, чтобы избежать статистических аномалий - Лимит редких матчей увеличен до 10, так как суммарный объем данных больше
create_view_query = """
CREATE OR REPLACE VIEW public.v_champion_power_curve_all AS
WITH match_intervals AS (
    SELECT 
        p.champion_id,
        nc.champion_name,
        p.win,
        CASE 
            WHEN m.game_duration / 60 < 20 THEN '1. <20 мин (FF/Сдались)'
            WHEN m.game_duration / 60 >= 20 AND m.game_duration / 60 < 25 THEN '2. 20-25 мин (Ранняя)'
            WHEN m.game_duration / 60 >= 25 AND m.game_duration / 60 < 30 THEN '3. 25-30 мин (Мид-гейм)'
            WHEN m.game_duration / 60 >= 30 AND m.game_duration / 60 < 35 THEN '4. 30-35 мин (Лейт-гейм)'
            ELSE '5. 35+ мин (Глубокий лейт)'
        END AS game_duration_interval
    FROM public.lol_players_matches AS p
    JOIN public.lol_matches AS m ON m.match_id = p.match_id
    JOIN public.nsi_champions AS nc ON nc.champion_id = p.champion_id
),
aggregated_stats AS (
    SELECT 
        champion_id,
        champion_name,
        game_duration_interval,
        COUNT(*) AS total_games,
        COUNT(CASE WHEN win = true THEN 1 END) AS wins_count
    FROM match_intervals
    GROUP BY champion_id, champion_name, game_duration_interval
)
SELECT 
    champion_id,
    champion_name,
    game_duration_interval,
    total_games,
    wins_count,
    ROUND((wins_count::numeric / total_games) * 100, 2) AS winrate
FROM aggregated_stats
WHERE total_games >= 10;
"""

In [30]:
create_supabase_view(engine, create_view_query)

'🎉 Представление успешно создано в Supabase через SQLAlchemy!'

## Вывод

Выполнен Этап T (Transform) в архитектуре ETL проекта:

1. **Предобработка данных (Python / Pandas)**: Полученные на этапе извлечения «сырые» данные в формате CSV очищены от дубликатов, в них обработаны пропуски, осуществлено приведение типов к нужным форматам. Финальные очищенные данные сохранены в промежуточные датафреймы.

Выполнена 1-я часть этапа L (Load) в архитектуре ETL проекта - загрузка в хранилище.
   
2. **Миграция в СУБД**: Очищенные структурированные данные перенесены в реляционные таблицы облачной базы данных Supabase (PostgreSQL) (ресурс https://supabase.com) для долгосрочного структурированного хранения.
3. **Агрегация и расчет KPI (SQL Views)**: С помощью SQL-скриптов на стороне базы данных сформированы витрины данных (представления). В них заранее рассчитаны сложные игровые метрики (KDA, индекс агрессии игроков, винрейты сторон по регионам), что позволяет минимизировать нагрузку на дашборд и обеспечить мгновенную загрузку интерфейса.